# A diffusion image generator

_Capstones_

**Stitch four papers into one small conditional diffusion model that denoises pure noise into toy images — and steer what it draws with classifier-free guidance.**

---

This notebook is a practice scaffold for the **A diffusion image generator** lesson. We build a tiny conditional DDPM on 2-D "toy images": a ring of Gaussian clusters whose colour/class is the condition.

The four-paper thread is:

| Paper idea | What we keep in this tiny build |
|---|---|
| VAE / latent generative modeling | Generate in a simple continuous space (2-D points instead of pixels). |
| U-Net | A down -> bottleneck -> up denoiser with a skip connection. |
| DDPM | Fixed forward noising, noise-MSE training, reverse denoising. |
| Classifier-free guidance (CFG) | Drop labels during training, then mix conditional and null predictions at sampling time. |

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders/dropdowns; without widgets they run a sensible fallback. Run top to bottom, then change the knobs. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib / torch ship with Colab.
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## First, look at the data

Before training a denoiser, look at the distribution it must learn. Our "images" are just 2-D points, so every important diffusion idea can be drawn as a scatter plot.

### Step 1 — The toy ring dataset

There are 8 Gaussian clusters on a radius-2 ring. Clusters whose center has `x < 0` are class 1 (left half); the others are class 0 (right half). The class label is the conditioning signal.

In [ ]:
# --- The toy data: a ring of 8 clusters; right half = class 0, left half = class 1. ---
def sample_data(n):
    k = torch.randint(0, 8, (n,))                  # which of the 8 clusters
    ang = k.float() / 8 * 2 * math.pi
    centers = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
    x = centers + torch.randn(n, 2) * 0.15
    c = (centers[:, 0] < 0).long()                 # class 1 if cluster is on the left (x<0), else class 0
    return x, c

torch.manual_seed(0)
x_data, c_data = sample_data(1200)

data_df = pd.DataFrame({
    "x": x_data[:12, 0].numpy().round(3),
    "y": x_data[:12, 1].numpy().round(3),
    "class": c_data[:12].numpy(),
})
print("data tensor:", tuple(x_data.shape), " labels:", tuple(c_data.shape))
display(data_df)

### Step 2 — Scatter the class-coloured clusters

Diffusion will learn this shape by seeing noisy versions of these points. The scatter plot is the target distribution: class 0 on the right, class 1 on the left.

In [ ]:
colors = np.where(c_data.numpy() == 1, "#ff7b72", "#79c0ff")
plt.figure(figsize=(5, 5))
plt.scatter(x_data[:, 0], x_data[:, 1], c=colors, s=10, alpha=0.65, edgecolors="none")
plt.axvline(0, color="k", lw=0.7, alpha=0.35)
plt.gca().set_aspect("equal")
plt.xlim(-3, 3); plt.ylim(-3, 3)
plt.title("toy data: ring clusters")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

balance = pd.Series(c_data.numpy()).value_counts().sort_index().rename_axis("class").reset_index(name="count")
display(balance)
plt.figure(figsize=(4.5, 2.7))
plt.bar(balance["class"].astype(str), balance["count"], color=["#79c0ff", "#ff7b72"])
plt.title("class balance")
plt.xlabel("class"); plt.ylabel("count")
plt.show()

### Step 3 — The DDPM variance schedule

The forward process is fixed: at each timestep it adds a small variance `beta_t`. The cumulative product `alpha_bar_t` tells us how much of the original signal survives after jumping directly to timestep `t`.

In [ ]:
# --- The forward noising schedule (paper-ddpm). A fixed recipe, no learning. ---
T = 50
betas = torch.linspace(1e-4, 0.02, T)              # small linear variance schedule
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                    # alpha-bar_t = prod_{s<=t} alpha_s

sched_df = pd.DataFrame({
    "t": [0, 1, 10, 25, 49],
    "beta_t": betas[[0, 1, 10, 25, 49]].numpy().round(5),
    "alpha_bar_t": abar[[0, 1, 10, 25, 49]].numpy().round(5),
})
display(sched_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(betas.numpy(), color="#79c0ff")
ax1.set_title("beta schedule")
ax1.set_xlabel("t"); ax1.set_ylabel("beta_t")
ax2.plot(abar.numpy(), color="#7ee787")
ax2.set_title("signal left: alpha_bar")
ax2.set_xlabel("t"); ax2.set_ylabel("alpha_bar_t")
plt.tight_layout(); plt.show()

### Step 4 — Forward noising: the ring melts into a blob

DDPM training does not run the forward chain step by step. It jumps to any timestep with

`x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps`, where `eps ~ N(0, I)`.

Below the same data distribution is shown after different amounts of noise.

In [ ]:
def q_sample(x0, t, eps=None):
    if eps is None:
        eps = torch.randn_like(x0)
    ab = abar[t].reshape(-1, 1) if torch.as_tensor(t).ndim else abar[int(t)]
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

torch.manual_seed(1)
base_x, base_c = sample_data(900)
eps_vis = torch.randn_like(base_x)
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharex=True, sharey=True)
for ax, t in zip(axes, [0, 10, 25, 49]):
    xt = q_sample(base_x, t, eps_vis)
    ax.scatter(xt[:, 0], xt[:, 1], c=np.where(base_c.numpy() == 1, "#ff7b72", "#79c0ff"),
               s=8, alpha=0.55, edgecolors="none")
    ax.set_title(f"t={t}")
    ax.set_aspect("equal"); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.suptitle("forward noising")
plt.tight_layout(); plt.show()

**🎛️ Play with it — forward noising timestep**

Drag `t` to choose how much signal survives. **Try:** `t=0`, `t=25`, `t=49`. **Watch:** cluster structure fades and the cloud becomes more Gaussian. **Why it matters:** the model is trained to undo exactly these noisy inputs.

In [ ]:
def play(t=25):
    torch.manual_seed(2)
    x0, c = sample_data(700)
    eps = torch.randn_like(x0)
    xt = q_sample(x0, int(t), eps)
    plt.figure(figsize=(5, 5))
    plt.scatter(xt[:, 0], xt[:, 1], c=np.where(c.numpy() == 1, "#ff7b72", "#79c0ff"),
                s=9, alpha=0.6, edgecolors="none")
    plt.gca().set_aspect("equal"); plt.xlim(-3.5, 3.5); plt.ylim(-3.5, 3.5)
    plt.title(f"forward noising at t={int(t)}")
    plt.xlabel("x_t[0]"); plt.ylabel("x_t[1]")
    plt.show()
    print("alpha_bar_t =", round(float(abar[int(t)]), 4))

try:
    from ipywidgets import interact, IntSlider
    interact(play, t=IntSlider(value=25, min=0, max=T-1, step=1))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## Reference implementation — PyTorch

We now build the conditional DDPM one small part at a time:

1. build the central forward-noising operation tensor by tensor;
2. package the U-Net-shaped MLP denoiser;
3. train it with the noise-MSE objective and label dropout;
4. sample by reverse denoising with classifier-free guidance.

## The hero operation, built tensor-by-tensor

The central DDPM trick is the reparameterized jump to any noise level:

`x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps`.

We will do it on a concrete toy batch before hiding it inside the training loop.

### Step 5 — Pick a concrete clean batch

Take a tiny batch of real points and labels. The shapes are deliberately small so you can trace every tensor.

In [ ]:
torch.manual_seed(3)
x0_hero, c_hero = sample_data(8)
t_hero = torch.full((x0_hero.shape[0],), 25, dtype=torch.long)

hero_df = pd.DataFrame({
    "row": range(len(x0_hero)),
    "x0_x": x0_hero[:, 0].numpy().round(3),
    "x0_y": x0_hero[:, 1].numpy().round(3),
    "class": c_hero.numpy(),
    "t": t_hero.numpy(),
})
display(hero_df)
print("x0:", tuple(x0_hero.shape), " c:", tuple(c_hero.shape), " t:", tuple(t_hero.shape))

### Step 6 — Sample the Gaussian noise `eps`

The model's training label is the exact noise vector used here. Each 2-D point gets its own 2-D `eps`.

In [ ]:
torch.manual_seed(4)
eps_hero = torch.randn_like(x0_hero)

eps_df = pd.DataFrame({
    "row": range(len(eps_hero)),
    "eps_x": eps_hero[:, 0].numpy().round(3),
    "eps_y": eps_hero[:, 1].numpy().round(3),
})
display(eps_df)
print("eps:", tuple(eps_hero.shape), " same shape as x0:", tuple(eps_hero.shape) == tuple(x0_hero.shape))

### Step 7 — Look up `alpha_bar_t` and its two square-root weights

For `t=25`, one scalar controls the clean-signal weight and one controls the noise weight. They are broadcast across the two coordinates of each point.

In [ ]:
ab_hero = abar[t_hero].unsqueeze(1)
clean_w = ab_hero.sqrt()
noise_w = (1 - ab_hero).sqrt()

weights_df = pd.DataFrame({
    "row": range(len(x0_hero)),
    "alpha_bar_t": ab_hero.squeeze(1).numpy().round(4),
    "sqrt(alpha_bar_t)": clean_w.squeeze(1).numpy().round(4),
    "sqrt(1-alpha_bar_t)": noise_w.squeeze(1).numpy().round(4),
})
display(weights_df.head())
print("alpha lookup:", tuple(ab_hero.shape), " broadcasts with x0:", tuple(x0_hero.shape))

### Step 8 — Form `x_t` and compare clean vs noisy points

Now multiply and add: a shrunken clean point plus scaled Gaussian noise. This is the noising operation the denoiser learns to invert.

In [ ]:
xt_hero = clean_w * x0_hero + noise_w * eps_hero

shape_df = pd.DataFrame([
    ("x0", tuple(x0_hero.shape)),
    ("eps", tuple(eps_hero.shape)),
    ("alpha weights", tuple(clean_w.shape)),
    ("xt", tuple(xt_hero.shape)),
], columns=["tensor", "shape"])
display(shape_df)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax0.scatter(x0_hero[:, 0], x0_hero[:, 1], c=np.where(c_hero.numpy() == 1, "#ff7b72", "#79c0ff"), s=80)
ax0.set_title("clean x0")
ax1.scatter(xt_hero[:, 0], xt_hero[:, 1], c=np.where(c_hero.numpy() == 1, "#ff7b72", "#79c0ff"), s=80)
ax1.set_title("noisy xt")
for ax in (ax0, ax1):
    ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

**🎛️ Play with it — one point plus noise**

A single point is easier to reason about than a cloud. **Try:** raise `noise_scale`. **Watch:** the red arrow `eps` grows and the point moves away from its clean location. **Why it matters:** the whole training target is "predict that arrow."

In [ ]:
point0 = torch.tensor([[2.0, 0.0]])
eps0 = torch.tensor([[-0.8, 1.1]])

def play(noise_scale=0.5):
    ns = float(noise_scale)
    noisy = point0 + ns * eps0
    plt.figure(figsize=(4.8, 4.8))
    plt.scatter(point0[:, 0], point0[:, 1], s=120, color="#79c0ff", label="x0")
    plt.scatter(noisy[:, 0], noisy[:, 1], s=120, color="#ff7b72", label="x0 + scale eps")
    plt.arrow(float(point0[0,0]), float(point0[0,1]), float(ns*eps0[0,0]), float(ns*eps0[0,1]),
              head_width=0.08, color="#ff7b72", length_includes_head=True)
    plt.xlim(0.5, 2.5); plt.ylim(-0.5, 1.5); plt.gca().set_aspect("equal")
    plt.grid(alpha=0.25); plt.legend(); plt.title("one point and eps")
    plt.show()
    print("noisy point:", noisy.numpy().round(3).tolist())

try:
    from ipywidgets import interact, FloatSlider
    interact(play, noise_scale=FloatSlider(value=0.5, min=0.0, max=1.5, step=0.05))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 9 — Reverse step and CFG in words

At sampling time we start from `x_T ~ N(0, I)`. For each timestep, the network predicts the noise still present. DDPM Algorithm 2 converts that prediction into a slightly cleaner mean. CFG runs the same network twice:

`eps_guided = (1 + w) eps_conditional - w eps_null`.

`w=0` uses the conditional model as-is; larger `w` exaggerates features that distinguish the chosen class from the null prediction.

### Step 10 — Define the U-Net-shaped noise predictor

This is a tiny MLP with a U-Net silhouette: down, bottleneck, up, and a skip connection from down to up. The conditioning label is embedded; row `2` is the NULL token used for label dropout and CFG.

In [ ]:
# --- The noise predictor eps_theta(x_t, t, c): a U-NET-shaped MLP (paper-unet) ---
#     with a class embedding that has a NULL row (paper-cfg). down -> bottleneck -> up + skip.
NULL = 2                                            # ids 0,1 = classes; id 2 = null token (no condition)

class UNetDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Embedding(3, 16)              # 3 rows: class 0, class 1, NULL
        self.down = nn.Sequential(nn.Linear(2 + 1 + 16, h), nn.SiLU())   # encoder (down)
        self.mid = nn.Sequential(nn.Linear(h, h), nn.SiLU())            # bottleneck
        self.up = nn.Sequential(nn.Linear(h + h, h), nn.SiLU())         # decoder (up), takes the skip
        self.out = nn.Linear(h, 2)

    def forward(self, x, t, c):
        te = (t.float() / T).unsqueeze(1)           # timestep, scaled to [0,1]
        cond = self.emb(c)                          # class embedding (or NULL)
        d = self.down(torch.cat([x, te, cond], 1))
        m = self.mid(d)
        u = self.up(torch.cat([m, d], 1))           # skip connection: hand 'd' across to the decoder
        return self.out(u)

### Step 11 — Inspect the denoiser's tensors and parameters

A denoiser input is a noisy 2-D point, a timestep, and a class id. The output has the same shape as the noise label: `(batch, 2)`.

In [ ]:
torch.manual_seed(0)
net_probe = UNetDenoiser()
with torch.no_grad():
    pred_probe = net_probe(xt_hero, t_hero, c_hero)

trace_df = pd.DataFrame([
    ("noisy points x_t", tuple(xt_hero.shape)),
    ("timesteps t", tuple(t_hero.shape)),
    ("class labels c", tuple(c_hero.shape)),
    ("predicted eps", tuple(pred_probe.shape)),
], columns=["thing", "shape"])
display(trace_df)

component_counts = {}
for name, p in net_probe.named_parameters():
    component = name.split(".")[0]
    component_counts[component] = component_counts.get(component, 0) + p.numel()
total_params = sum(component_counts.values())
param_df = pd.DataFrame({
    "component": list(component_counts),
    "parameters": list(component_counts.values()),
    "% of total": [round(100 * v / total_params, 1) for v in component_counts.values()],
})
display(param_df)
print("total trainable parameters:", f"{total_params:,}")

**🎛️ Play with it — beta schedule endpoints**

The schedule is fixed before training. **Try:** increase `beta_end`. **Watch:** `alpha_bar` falls faster, meaning less signal survives at large `t`. **Why it matters:** too little noise makes sampling weak; too much can make denoising hard.

In [ ]:
def play(beta_start=1e-4, beta_end=0.02):
    bs = float(beta_start); be = float(beta_end)
    if bs <= 0 or be <= 0 or bs >= be:
        print("Choose 0 < beta_start < beta_end."); return
    b = torch.linspace(bs, be, T)
    ab = torch.cumprod(1 - b, 0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.1))
    ax1.plot(b.numpy(), color="#79c0ff"); ax1.set_title("beta_t")
    ax1.set_xlabel("t")
    ax2.plot(ab.numpy(), color="#7ee787"); ax2.set_title("alpha_bar_t")
    ax2.set_xlabel("t"); ax2.set_ylim(0, 1.02)
    plt.tight_layout(); plt.show()
    print("final alpha_bar:", round(float(ab[-1]), 4))

try:
    from ipywidgets import interact, FloatSlider
    interact(play,
             beta_start=FloatSlider(value=1e-4, min=1e-5, max=0.005, step=1e-4, readout_format=".4f"),
             beta_end=FloatSlider(value=0.02, min=0.005, max=0.08, step=0.005))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 12 — One training step before the full run

Before the long loop, do one optimizer step on one batch and re-measure the same batch. Because the target is the sampled `eps`, the loss is a mean squared error between true noise and predicted noise.

In [ ]:
p_uncond = 0.2

def make_training_batch(n=512):
    x0, c = sample_data(n)
    drop = torch.rand(n) < p_uncond                        # paper-cfg Alg. 1: with prob p_uncond...
    c_in = torch.where(drop, torch.full_like(c, NULL), c)   # ...replace the label by the NULL token
    tb = torch.randint(0, T, (n,))                          # random timesteps (paper-ddpm Alg. 1)
    eps = torch.randn_like(x0)                              # the noise to predict
    ab = abar[tb].unsqueeze(1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps             # Eq. 4: forward-noise in one shot (reparam.)
    return xt, tb, c_in, eps

torch.manual_seed(5)
one_net = UNetDenoiser()
one_opt = torch.optim.Adam(one_net.parameters(), lr=1e-3)
xtb, tb, cb, epsb = make_training_batch(512)
before = ((epsb - one_net(xtb, tb, cb)) ** 2).mean()
one_opt.zero_grad(); before.backward(); one_opt.step()
after = ((epsb - one_net(xtb, tb, cb)) ** 2).mean()
print(f"noise-MSE on this batch: before = {before.item():.4f} -> after one step = {after.item():.4f}")

### Step 13 — Train: noise-MSE with label dropout

This is DDPM's simple objective plus CFG's label-dropping trick. We start from a fresh model and record the loss curve for the results section. (Our small run; exact values vary by hardware and seed.)

In [ ]:
def train():
    torch.manual_seed(0)
    net = UNetDenoiser()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_hist = []
    for step in range(4000):
        x0, c = sample_data(512)
        drop = torch.rand(512) < p_uncond                        # paper-cfg Alg. 1: with prob p_uncond...
        c_in = torch.where(drop, torch.full_like(c, NULL), c)    # ...replace the label by the NULL token
        tb = torch.randint(0, T, (512,))                         # random timesteps (paper-ddpm Alg. 1)
        eps = torch.randn_like(x0)                               # the noise to predict
        ab = abar[tb].unsqueeze(1)
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps              # Eq. 4: forward-noise in one shot (reparam.)
        pred = net(xt, tb, c_in)
        loss = ((eps - pred) ** 2).mean()                        # Eq. 14: L_simple (predict the noise)
        opt.zero_grad()
        loss.backward()
        opt.step()
        loss_hist.append((step, float(loss.item())))
        if step % 1000 == 0:
            print(f"  step {step:4d}  loss {loss.item():.4f}")
    return net, loss_hist

print("TRAIN the conditional DDPM:")
net, loss_hist = train()

### Step 14 — Define guided reverse sampling

Sampling reverses the noising process. Each reverse step runs the network twice: once with the requested class and once with NULL, then mixes the two noise estimates using guidance scale `w`.

In [ ]:
# --- Guided sampling (paper-ddpm Alg. 2 + paper-cfg Eq. 6): two predictions/step, mix, step. ---
@torch.no_grad()
def sample(net, n, c_val, w, snapshot_at=(), steps=None):
    steps = T if steps is None else int(steps)
    steps = max(1, min(T, steps))
    timeline = torch.linspace(T - 1, 0, steps).round().long().unique(sorted=True)
    timeline = torch.flip(timeline, dims=[0])       # descending, e.g. 49 -> ... -> 0
    if timeline[-1].item() != 0:
        timeline = torch.cat([timeline, torch.tensor([0])])

    x = torch.randn(n, 2)                                       # x_T ~ N(0, I): pure noise
    c = torch.full((n,), c_val, dtype=torch.long)
    cn = torch.full((n,), NULL, dtype=torch.long)               # null token for the label-blind pass
    snaps = {}
    requested = set(int(t) for t in snapshot_at)
    for i, t_tensor in enumerate(timeline):
        t = int(t_tensor.item())
        tb = torch.full((n,), t, dtype=torch.long)
        ec = net(x, tb, c)                                      # conditional   eps_theta(x, t, c)
        eu = net(x, tb, cn)                                     # unconditional eps_theta(x, t)
        e = (1 + w) * ec - w * eu                               # paper-cfg Eq. 6: guided noise estimate
        a = alphas[t]
        ab = abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)   # paper-ddpm Alg. 2 denoising step
        if i < len(timeline) - 1 and t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean                                            # no noise added at the last step
        if t in requested:
            snaps[t] = x.clone()
    return x, snaps

# Cluster centers for evaluation/plots.
ang = torch.arange(8).float() / 8 * 2 * math.pi
modes = torch.stack([torch.cos(ang), torch.sin(ang)], 1) * 2.0
left_modes = modes[modes[:, 0] < 0]
right_modes = modes[modes[:, 0] >= 0]

**🎛️ Play with it — guidance scale `w`**

Sample class-1 points with different CFG strengths. **Try:** `w=0`, `w=1`, `w=3`. **Watch:** higher guidance pulls more points onto the left-half class clusters. **Why it matters:** CFG is the steering knob used by modern diffusion systems.

In [ ]:
def play(w=3.0):
    torch.manual_seed(10)
    s, _ = sample(net, 500, c_val=1, w=float(w))
    d_left = torch.cdist(s, left_modes).min(1).values
    frac_left = (s[:, 0] < 0).float().mean().item()
    on_mode = (d_left < 0.45).float().mean().item()
    plt.figure(figsize=(5, 5))
    plt.scatter(s[:, 0], s[:, 1], s=10, alpha=0.55, color="#ff7b72", edgecolors="none")
    plt.scatter(left_modes[:, 0], left_modes[:, 1], marker="x", s=100, color="black", label="class 1 centers")
    plt.axvline(0, color="k", lw=0.7, alpha=0.35)
    plt.gca().set_aspect("equal"); plt.xlim(-3.2, 3.2); plt.ylim(-3.2, 3.2)
    plt.title(f"class 1 samples, w={float(w):.1f}")
    plt.legend(); plt.show()
    print("frac on left half:", round(frac_left, 3), "  frac near class-1 mode:", round(on_mode, 3))

try:
    from ipywidgets import interact, FloatSlider
    interact(play, w=FloatSlider(value=3.0, min=0.0, max=5.0, step=0.25))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — number of reverse steps**

A full DDPM uses many tiny denoising steps. **Try:** fewer steps. **Watch:** clouds are blurrier or less settled when the reverse chain is shortened. **Why it matters:** sampling speed and sample quality are a core diffusion tradeoff.

In [ ]:
def play(steps=50):
    torch.manual_seed(11)
    s, _ = sample(net, 450, c_val=1, w=3.0, steps=int(steps))
    d_left = torch.cdist(s, left_modes).min(1).values
    plt.figure(figsize=(5, 5))
    plt.scatter(s[:, 0], s[:, 1], s=10, alpha=0.55, color="#ff7b72", edgecolors="none")
    plt.scatter(left_modes[:, 0], left_modes[:, 1], marker="x", s=100, color="black")
    plt.axvline(0, color="k", lw=0.7, alpha=0.35)
    plt.gca().set_aspect("equal"); plt.xlim(-3.2, 3.2); plt.ylim(-3.2, 3.2)
    plt.title(f"reverse steps = {int(steps)}")
    plt.show()
    print("mean distance to class-1 centers:", round(float(d_left.mean()), 3))

try:
    from ipywidgets import interact, IntSlider
    interact(play, steps=IntSlider(value=50, min=5, max=T, step=5))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — choose the class label**

The same trained denoiser can draw either condition. **Try:** class 0 and class 1. **Watch:** class 0 lands on the right half; class 1 lands on the left half. **Why it matters:** the label embedding is the steering input.

In [ ]:
def play(c_val=1):
    c_int = int(c_val)
    torch.manual_seed(12)
    s, _ = sample(net, 500, c_val=c_int, w=3.0)
    target = left_modes if c_int == 1 else right_modes
    color = "#ff7b72" if c_int == 1 else "#79c0ff"
    plt.figure(figsize=(5, 5))
    plt.scatter(s[:, 0], s[:, 1], s=10, alpha=0.55, color=color, edgecolors="none")
    plt.scatter(target[:, 0], target[:, 1], marker="x", s=100, color="black", label="target centers")
    plt.axvline(0, color="k", lw=0.7, alpha=0.35)
    plt.gca().set_aspect("equal"); plt.xlim(-3.2, 3.2); plt.ylim(-3.2, 3.2)
    plt.title(f"samples for class {c_int}")
    plt.legend(); plt.show()

try:
    from ipywidgets import interact, Dropdown
    interact(play, c_val=Dropdown(options=[0, 1], value=1, description="class"))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — guided vs unguided**

Compare the same random seed with `w=0` and `w=3`. **Try:** re-run after training finishes. **Watch:** the guided cloud is usually tighter and more class-specific. **Why it matters:** this isolates the CFG mixing term.

In [ ]:
def play(c_val=1):
    c_int = int(c_val)
    target = left_modes if c_int == 1 else right_modes
    color = "#ff7b72" if c_int == 1 else "#79c0ff"
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
    for ax, w in zip(axes, [0.0, 3.0]):
        torch.manual_seed(13)
        s, _ = sample(net, 450, c_val=c_int, w=w)
        ax.scatter(s[:, 0], s[:, 1], s=10, alpha=0.55, color=color, edgecolors="none")
        ax.scatter(target[:, 0], target[:, 1], marker="x", s=90, color="black")
        ax.axvline(0, color="k", lw=0.7, alpha=0.35)
        ax.set_aspect("equal"); ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
        ax.set_title(f"w={w:.0f}")
    plt.suptitle(f"class {c_int}: unguided vs guided")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, Dropdown
    interact(play, c_val=Dropdown(options=[0, 1], value=1, description="class"))
except Exception:
    play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## Visualize the data & results

_Does the assembled conditional DDPM denoise pure noise into the ring of clusters, and does raising the guidance scale w pull class-1 samples onto their (left-half) clusters?_

We answer with the trained model and loss history already recorded above — no second training run.

### Step 15 — Training loss curve

The noise-MSE should fall as the denoiser learns to predict `eps` from `x_t`, `t`, and the condition/null token.

In [ ]:
steps = [s for s, _ in loss_hist]
losses = [l for _, l in loss_hist]
plt.figure(figsize=(7.5, 3.4))
plt.plot(steps, losses, color="#7ee787", lw=1.2)
plt.xlabel("training step"); plt.ylabel("noise MSE")
plt.title("training loss")
plt.show()
print("first loss:", round(losses[0], 4), " final loss:", round(losses[-1], 4))

### Step 16 — Noise to ring progression

Start from pure Gaussian noise and snapshot the class-1 guided reverse chain at `t=49`, `t=20`, and `t=0`. The point cloud moves from blob to ring clusters.

In [ ]:
torch.manual_seed(7)
_, snaps = sample(net, 800, c_val=1, w=3.0, snapshot_at=(49, 20, 0))
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5), sharex=True, sharey=True)
for ax, t in zip(axes, [49, 20, 0]):
    pts = snaps[t]
    ax.scatter(pts[:, 0], pts[:, 1], s=9, alpha=0.55, color="#ff7b72", edgecolors="none")
    ax.scatter(left_modes[:, 0], left_modes[:, 1], marker="x", s=90, color="black")
    ax.axvline(0, color="k", lw=0.7, alpha=0.35)
    ax.set_title(f"t={t}")
    ax.set_aspect("equal"); ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
plt.suptitle("reverse denoising: class 1, w=3")
plt.tight_layout(); plt.show()

radius_df = pd.DataFrame({
    "t": [49, 20, 0],
    "mean radius": [round(float(snaps[t].norm(dim=1).mean()), 3) for t in [49, 20, 0]],
})
display(radius_df)

### Step 17 — Guidance sweep metrics

For class 1, the correct clusters are on the left. As `w` grows, samples should land more often on the left half and closer to the left-half centers.

In [ ]:
rows = []
for w in [0.0, 1.0, 3.0]:
    torch.manual_seed(20)
    s, _ = sample(net, 1200, c_val=1, w=w)
    d_left = torch.cdist(s, left_modes).min(1).values
    rows.append({
        "w": w,
        "frac on class-1 half": round(float((s[:, 0] < 0).float().mean()), 3),
        "frac near class-1 mode": round(float((d_left < 0.45).float().mean()), 3),
        "mean dist to class-1 mode": round(float(d_left.mean()), 3),
    })
guidance_df = pd.DataFrame(rows)
display(guidance_df)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(guidance_df["w"], guidance_df["frac on class-1 half"], marker="o", label="left half")
ax.plot(guidance_df["w"], guidance_df["frac near class-1 mode"], marker="o", label="near mode")
ax.set_ylim(0, 1.02); ax.set_xlabel("guidance scale w"); ax.set_ylabel("fraction")
ax.set_title("guidance improves class-1 targeting")
ax.legend(); plt.show()

### Step 18 — Before/after: unguided vs strong guidance

The left panel is `w=0`; the right is `w=3`. Strong guidance should concentrate the class-1 samples on the target half and modes. These are our-run visuals, not paper numbers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for ax, w in zip(axes, [0.0, 3.0]):
    torch.manual_seed(21)
    s, _ = sample(net, 700, c_val=1, w=w)
    ax.scatter(s[:, 0], s[:, 1], s=10, alpha=0.55, color="#ff7b72", edgecolors="none")
    ax.scatter(left_modes[:, 0], left_modes[:, 1], marker="x", s=90, color="black", label="class 1 centers")
    ax.axvline(0, color="k", lw=0.7, alpha=0.35)
    ax.set_aspect("equal"); ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
    ax.set_title(f"w={w:.0f}")
plt.suptitle("before/after guidance")
plt.tight_layout(); plt.show()

### Practice

1. Change the class dropout probability `p_uncond`. What happens if it is `0.0`? What happens if it is very high?
2. Increase `T` or change the beta schedule. Does sampling become smoother or harder to train?
3. Make the toy dataset harder: more clusters, different radii, or a spiral. Which parts of the notebook need to change?
4. Replace the U-Net-shaped MLP with a wider hidden size. Compare parameter count, loss curve, and sample quality.